In [2]:
import pandas as pd
import numpy as np
from itertools import product
import plotly.graph_objects as go
import nbformat

# Load the Excel file
file_path = r"C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey diagram table.xlsx"
df = pd.read_excel(file_path, sheet_name='RAW')

print("Original data shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())

Original data shape: (56, 4)

First few rows:
                                    selected article  \
0  Enhancing plant stress diagnosis using thermal...   
1  Deep learning-based plant classification and c...   
2  Cassava disease detection using a lightweight ...   
3  CSXAI: a lightweight 2D CNN-SVM model for dete...   
4  DBA-ViNet: an effective deep learning framewor...   

                                 Crops Name  \
0                                      Rice   
1                                      Rice   
2                                   Cassava   
3        Peach, Cherry, Soybean, Strawberry   
4  Apple, Guava, Mango, Orange, Pomegranate   

                                     Model Type Used  \
0         GCViT, VGG-16, VGG-19, ResNet-50, Xception   
1              PlantDXAI (CNN + Discriminator / GAN)   
2        CDDNet (Lightweight soft attention network)   
3                     CSXAI (Lightweight 2D CNN-SVM)   
4  DBA-ViNet, EfficientNetV2, ConvNeXt, YOLOv8, M...  

In [3]:
# Function to clean and split comma-separated values
def split_and_clean(value):
    """Split on commas only when they are outside parentheses."""
    if pd.isna(value):
        return []

    text = str(value).strip()
    parts = []
    current = []
    depth = 0

    for char in text:
        if char == '(':
            depth += 1
            current.append(char)
        elif char == ')':
            depth = max(depth - 1, 0)
            current.append(char)
        elif char == ',' and depth == 0:
            item = ''.join(current).strip()
            if item:
                parts.append(item)
            current = []
        else:
            current.append(char)

    item = ''.join(current).strip()
    if item:
        parts.append(item)

    return parts

# Expand the data: create all combinations per article
expanded_records = []

for idx, row in df.iterrows():
    article = row['selected article']
    crops = split_and_clean(row['Crops Name'])
    models = split_and_clean(row['Model Type Used'])
    xai_methods = split_and_clean(row['XAI Methods Used'])
    
    # Create all combinations (cartesian product)
    for crop, model, xai in product(crops, models, xai_methods):
        expanded_records.append({
            'Crops Name': crop,
            'Model Type Used': model,
            'XAI Methods Used': xai,
            'article_count': 1  # Each combination from an article counts as 1
        })

# Create expanded dataframe
expanded_df = pd.DataFrame(expanded_records)

print(f"Total articles: {len(df)}")
print(f"Total expanded records: {len(expanded_df)}")
print(f"\nUnique Crops: {expanded_df['Crops Name'].nunique()}")
print(f"Unique Models: {expanded_df['Model Type Used'].nunique()}")
print(f"Unique XAI Methods: {expanded_df['XAI Methods Used'].nunique()}")
print(f"\nFirst 10 expanded records:")
print(expanded_df.head(10))

# Aggregate counts per flow
flow_counts = expanded_df.groupby(['Crops Name', 'Model Type Used', 'XAI Methods Used']).size().reset_index(name='count')
print(f"\nUnique flows: {len(flow_counts)}")
print(flow_counts.head(10))

# Save expanded_df and flow_counts into the same Excel file as new sheets
file_path = r"C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey diagram table.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    expanded_df.to_excel(writer, sheet_name="EXPANDED", index=False)
    flow_counts.to_excel(writer, sheet_name="FLOW_COUNTS", index=False)

print(f"\nData saved to Excel:")
print(f"- EXPANDED sheet rows: {len(expanded_df)}")
print(f"- FLOW_COUNTS sheet rows: {len(flow_counts)}")
print(f"- File: {file_path}")

Total articles: 56
Total expanded records: 337

Unique Crops: 43
Unique Models: 99
Unique XAI Methods: 21

First 10 expanded records:
  Crops Name Model Type Used     XAI Methods Used  article_count
0       Rice           GCViT  Activation Heatmaps              1
1       Rice           GCViT   ViT Attention Maps              1
2       Rice          VGG-16  Activation Heatmaps              1
3       Rice          VGG-16   ViT Attention Maps              1
4       Rice          VGG-19  Activation Heatmaps              1
5       Rice          VGG-19   ViT Attention Maps              1
6       Rice       ResNet-50  Activation Heatmaps              1
7       Rice       ResNet-50   ViT Attention Maps              1
8       Rice        Xception  Activation Heatmaps              1
9       Rice        Xception   ViT Attention Maps              1

Unique flows: 333
  Crops Name                         Model Type Used XAI Methods Used  count
0      Apple                                ConvNeXt   

# Improved visualization with color coding

In [3]:
import re
import pandas as pd
import numpy as np
import plotly.express as px
from matplotlib import colors as mcolors
import plotly.graph_objects as go

# Config / input
file_path = r"C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey diagram table.xlsx"
flow_counts = pd.read_excel(file_path, sheet_name="Final")

min_record_model = 1
min_record_xai = 1

crop_col = "Crops Name"
model_col = "Model Used"
xai_col = "XAI Methods Used"

# Aggregate counts per node
crop_counts = flow_counts.groupby(crop_col)["count"].sum()
model_counts = flow_counts.groupby(model_col)["count"].sum()
xai_counts = flow_counts.groupby(xai_col)["count"].sum()

# Filter by thresholds
kept_models = model_counts[model_counts >= min_record_model].index.tolist()
kept_xai = xai_counts[xai_counts >= min_record_xai].index.tolist()

# Build flows
crop_model_flows = (
    flow_counts[flow_counts[model_col].isin(kept_models)]
    .groupby([crop_col, model_col])["count"]
    .sum()
    .reset_index()
)

model_xai_flows = (
    flow_counts[
        flow_counts[model_col].isin(kept_models) & flow_counts[xai_col].isin(kept_xai)
    ]
    .groupby([model_col, xai_col])["count"]
    .sum()
    .reset_index()
)

# Sort nodes A→Z
crops_nodes = sorted(crop_counts.index.tolist())
models_nodes = sorted(kept_models)
xai_nodes = sorted(kept_xai)

# Labeled nodes with counts
crops_labeled = [f"{c} ({int(crop_counts[c])})" for c in crops_nodes]
models_labeled = [f"{m} ({int(model_counts[m])})" for m in models_nodes]
xai_labeled = [f"{x} ({int(xai_counts[x])})" for x in xai_nodes]
all_nodes = crops_labeled + models_labeled + xai_labeled

# Mappings from label -> index
crop_to_idx = {c: i for i, c in enumerate(crops_labeled)}
model_to_idx = {m: i + len(crops_labeled) for i, m in enumerate(models_labeled)}
xai_to_idx = {x: i + len(crops_labeled) + len(models_labeled) for i, x in enumerate(xai_labeled)}

# Ordered links
crop_model_flows = crop_model_flows.sort_values([crop_col, model_col]).reset_index(drop=True)
model_xai_flows = model_xai_flows.sort_values([model_col, xai_col]).reset_index(drop=True)

source_indices = []
target_indices = []
values = []

for _, r in crop_model_flows.iterrows():
    crop_label = f"{r[crop_col]} ({int(crop_counts[r[crop_col]])})"
    model_label = f"{r[model_col]} ({int(model_counts[r[model_col]])})"
    if crop_label in crop_to_idx and model_label in model_to_idx:
        source_indices.append(crop_to_idx[crop_label])
        target_indices.append(model_to_idx[model_label])
        values.append(int(r["count"]))

for _, r in model_xai_flows.iterrows():
    model_label = f"{r[model_col]} ({int(model_counts[r[model_col]])})"
    xai_label = f"{r[xai_col]} ({int(xai_counts[r[xai_col]])})"
    if model_label in model_to_idx and xai_label in xai_to_idx:
        source_indices.append(model_to_idx[model_label])
        target_indices.append(xai_to_idx[xai_label])
        values.append(int(r["count"]))

# Robust color converter
def color_to_rgba(col, alpha=0.95):
    try:
        s = str(col).strip()
        if s.startswith("rgb("):
            nums = list(map(int, re.findall(r"\d+", s)))[:3]
            return f"rgba({nums[0]},{nums[1]},{nums[2]},{alpha})"
        r, g, b = mcolors.to_rgb(s)
        return f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{alpha})"
    except Exception:
        return f"rgba(200,200,200,{alpha})"

# Discrete palettes for all groups
palette_crops = px.colors.qualitative.Set2
palette_models = px.colors.qualitative.Plotly
palette_xai = px.colors.qualitative.Dark24

def pick_palette(palette, n):
    if n <= 0:
        return []
    if n == 1:
        return [palette[0]]
    if n <= len(palette):
        return palette[:n]
    return [palette[i % len(palette)] for i in range(n)]

crops_colors = pick_palette(palette_crops, len(crops_nodes))
models_colors = pick_palette(palette_models, len(models_nodes))
xai_colors = pick_palette(palette_xai, len(xai_nodes))

node_colors = crops_colors + models_colors + xai_colors
node_colors_rgba = [color_to_rgba(c, 0.95) for c in node_colors]
link_colors = [
    color_to_rgba(node_colors[s], 0.35) if 0 <= s < len(node_colors) else color_to_rgba("#C0C0C0", 0.35)
    for s in source_indices
]

# Node positions
def column_positions(count, x_val):
    if count <= 0:
        return []
    if count == 1:
        return [(x_val, 0.5)]
    ys = np.linspace(0.05, 0.95, count)
    return [(x_val, y) for y in ys]

c_pos = column_positions(len(crops_nodes), 0.0)
m_pos = column_positions(len(models_nodes), 0.5)
x_pos = column_positions(len(xai_nodes), 1.0)

positions = c_pos + m_pos + x_pos
node_x = [p[0] for p in positions]
node_y = [p[1] for p in positions]

# Build Sankey
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=15,
        thickness=45,
        line=dict(color="rgba(0,0,0,0.6)", width=0.6),
        label=all_nodes,
        color=node_colors_rgba,
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=source_indices,
        target=target_indices,
        value=values,
        color=link_colors,
        hovertemplate="Flow: %{value}<extra></extra>"
    )
)])

# No figure title
fig.update_layout(
    font=dict(size=16),
    height=1200,
    width=1500,
    margin=dict(l=120, r=120, t=80, b=80)
)

fig.add_annotation(x=0.0, y=1.05, text="Crops", showarrow=False, xref="paper", yref="paper", font=dict(size=20))
fig.add_annotation(x=0.5, y=1.05, text="Models", showarrow=False, xref="paper", yref="paper", font=dict(size=20))
fig.add_annotation(x=1.0, y=1.05, text="XAI Methods", showarrow=False, xref="paper", yref="paper", font=dict(size=20))

fig.show()

# Save HTML
output_html = r"C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey_diagram.html"
fig.write_html(output_html)


print(f"Sankey saved at: {output_html}")
print(f"SVG saved at: {output_svg}")

Sankey saved at: C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey_diagram.html
SVG saved at: C:\Users\kamalesh\OneDrive\Articles\Review Article\4. ExplainableAI\codes\sankey diagram\sankey_diagram.svg
